# Amazon ML Challenge 2026 — Phase 0: Data Audit & Evaluation Foundation

This notebook provides visual and programmatic verification of the **Phase 0 Foundation**:
1. Strict TSV schema validation and immutable raw data loading.
2. Fact-based data audit (sizes, missingness, open-set country distributions, duplication).
3. Ground-truth structure analysis (singletons, multi-matches, cross-source links).
4. Descriptive raw string agreement diagnostics.
5. Entity-level validation split with zero leakage.
6. Exact Amazon-style Macro $F_{0.5}$ evaluation with numerical challenge verification.

> **Anti-Scope Reminder:** No blocking, candidate generation, ML matching models, or external lookups are implemented in Phase 0.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd

from src.utils.env import PROJECT_ROOT, resolve_train_dir, resolve_test_dir
from src.data.loader import load_source_tsv, load_ground_truth_tsv
from src.evaluation.metric import evaluate_entity_f05, evaluate_f05
from src.validation.split import create_entity_validation_split, partition_ground_truth
from src.validation.leakage import verify_no_leakage

train_dir = resolve_train_dir()
test_dir = resolve_test_dir()
print(f"Project Root: {PROJECT_ROOT}")
print(f"Train Directory: {train_dir}")
print(f"Test Directory:  {test_dir}")

## 1. Inspect Phase 0 Audit Results
The audit is pre-computed factually by the streaming `DataIntegrityChecker` across all 24 million records.

In [ ]:
audit_file = PROJECT_ROOT / "reports" / "phase0" / "data_audit.json"
if audit_file.is_file():
    with open(audit_file, "r", encoding="utf-8") as f:
        audit = json.load(f)
    print("=== TRAIN SOURCES SUMMARY ===")
    for src, info in audit["train_sources"].items():
        print(f"{src.upper()} ({info['file']}): {info['total_rows']:,} rows, {info['size_mb']} MB, Countries: {info['country_distribution']}")
        
    gt = audit["ground_truth"]
    print("\n=== GROUND TRUTH SUMMARY ===")
    print(f"Total S1 Entities:     {gt['total_rows']:,}")
    print(f"True Singletons:       {gt['singleton_count']:,} ({gt['singleton_percentage']}%)")
    print(f"One-Match Entities:    {gt['one_match_count']:,} ({gt['one_match_percentage']}%)")
    print(f"Multi-Match Entities:  {gt['multi_match_count']:,} ({gt['multi_match_percentage']}%)")
    print(f"Total True Links:      {gt['total_true_links']:,}")
    print(f"Mean Links / S1:       {gt['mean_links_per_s1']}")
    
    diag = audit["raw_exact_agreement_diagnostics"]
    print("\n=== RAW STRING EQUALITY DIAGNOSTICS (50k sample of true links) ===")
    print(f"Exact Name Agreement:            {diag['exact_name_equality_percentage']}%")
    print(f"Exact Address Agreement:         {diag['exact_address_equality_percentage']}%")
    print(f"Exact (Name + Address) Match:    {diag['exact_name_and_address_equality_percentage']}%")
else:
    print(f"Audit file not found at {audit_file}. Run `python -m src.data.integrity` first.")

## 2. Verify Exact Amazon Macro $F_{0.5}$ Evaluator
Confirming behavior on the challenge numerical example:
- Ground truth: `[S2-00047, S3-00812]`
- Prediction:   `[S2-00047, S2-00193, S3-00812]`
- Expected: Precision = 2/3, Recall = 1.0, $F_{0.5} = 5/7 \approx 0.714$

In [ ]:
f05, prec, rec = evaluate_entity_f05(
    true_ids=["S2-00047", "S3-00812"],
    predicted_ids=["S2-00047", "S2-00193", "S3-00812"]
)
print(f"Precision: {prec:.4f} (expected: 0.6667)")
print(f"Recall:    {rec:.4f} (expected: 1.0000)")
print(f"F_0.5:     {f05:.4f} (expected: 0.7143)")
assert round(f05, 3) == 0.714, "Challenge numerical example failed!"

## 3. Entity-Level Validation Split Verification
Demonstrating the deterministic 80/20 Source-1 entity split and zero-leakage guarantee.

In [ ]:
split_summary_file = PROJECT_ROOT / "reports" / "phase0" / "validation_split_summary.json"
if split_summary_file.is_file():
    with open(split_summary_file, "r", encoding="utf-8") as f:
        split_meta = json.load(f)
    print(json.dumps(split_meta, indent=2))
else:
    print("Run `python -m src.validation.split` to generate the validation split.")